In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Tue Dec 17 00:00:00 2023
@author: chun (refactored)
Add equalization
Rician channel
"""
import os
import re
import glob
import time
import torch
import yaml
import numpy as np
import torch.nn as nn
import torch.optim as optim
from fractions import Fraction
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from torch.nn.parallel import DataParallel
from tqdm import tqdm
from tensorboardX import SummaryWriter

from modelH import DeepJSCC, ratio2filtersize
from utils import image_normalization, set_seed, view_model_param
from dataset import Vanilla


def train_epoch(model, optimizer, param, data_loader):
    model.train()
    total_loss = 0.0
    for it, (images, _) in enumerate(data_loader):
        images = images.to(param['device'])
        optimizer.zero_grad()
        outputs = model(images)
        outputs = image_normalization('denormalization')(outputs)
        images = image_normalization('denormalization')(images)
        loss = model.loss(images, outputs) if not param['parallel'] else model.module.loss(images, outputs)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / (it + 1)
    return avg_loss


def evaluate_epoch(model, param, data_loader):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for it, (images, _) in enumerate(data_loader):
            images = images.to(param['device'])
            outputs = model(images)
            outputs = image_normalization('denormalization')(outputs)
            images = image_normalization('denormalization')(images)
            loss = (model.loss(images, outputs)
                    if not param['parallel'] else model.module.loss(images, outputs))
            total_loss += loss.item()
    avg_loss = total_loss / (it + 1)
    return avg_loss


def config_parser_pipeline():
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--dataset', default='eurosat', type=str,
                        choices=['cifar10', 'imagenet', 'eurosat'], help='Dataset name')
    parser.add_argument('--out', default='./out', type=str, help='Output path for logs and checkpoints')
    parser.add_argument('--disable_tqdm', action='store_true', help='Disable tqdm progress bars')
    parser.add_argument('--device', default='cuda:0', type=str, help='Device: cuda:0 / cpu')
    parser.add_argument('--parallel', action='store_true', help='Use DataParallel if multiple GPUs are available')
    parser.add_argument('--snr_list', default=['1', '4', '7', '13', '19'], nargs='+',
                        help='List of SNR values (e.g. 5 10 15)')
    parser.add_argument('--K_factor_list', default=['1', '3', '5', '7', '9'], nargs='+',
                        help='List of K_factor values (e.g. 5 10 15)')
    parser.add_argument('--ratio_list', default=['1/6', '1/12'], nargs='+',
                        help='List of channel ratios (e.g. 1/6 1/12)')
    parser.add_argument('--channel', default='Rician', type=str,
                        choices=['AWGN', 'Rayleigh', 'Rician'], help='Channel type')
    parser.add_argument('--equalization', type=bool, choices=[True, False], default=True, help='Use equalization or not')

    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"Ignoring unknown args: {unknown}")
    return args


def main_pipeline():
    args = config_parser_pipeline()
    args.snr_list = list(map(float, args.snr_list))
    args.K_factor_list = list(map(float, args.K_factor_list))
    args.ratio_list = [float(Fraction(x)) for x in args.ratio_list]

    print("📡 Training Start")
    for K_factor in args.K_factor_list:
        for ratio in args.ratio_list:
            for snr in args.snr_list:
                params = prepare_params(args, ratio, snr, K_factor)
                train_pipeline(params)


def prepare_params(args, ratio, snr, K_factor):
    params = {
        'disable_tqdm': args.disable_tqdm,
        'dataset': args.dataset,
        'out_dir': args.out,
        'device': args.device if torch.cuda.is_available() else 'cpu',
        'parallel': args.parallel,
        'snr': snr,
        'K_factor': K_factor,
        'ratio': ratio,
        'channel': args.channel,
        'equalization': args.equalization,
    }

    if args.dataset == 'cifar10':
        params.update({
            'batch_size': 64, 'num_workers': 4, 'epochs': 1000,
            'init_lr': 1e-3, 'weight_decay': 5e-4,
            'if_scheduler': True, 'step_size': 640, 'gamma': 0.1,
            'ReduceLROnPlateau': False, 'lr_reduce_factor': 0.5,
            'lr_schedule_patience': 15, 'min_lr': 1e-5, 'max_time': 12,
            'seed': 42,
        })
    elif args.dataset == 'imagenet':  # imagenet
        params.update({
            'batch_size': 32, 'num_workers': 4, 'epochs': 500,
            'init_lr': 1e-4, 'weight_decay': 5e-4,
            'if_scheduler': True, 'gamma': 0.1,
            'ReduceLROnPlateau': True, 'lr_reduce_factor': 0.5,
            'lr_schedule_patience': 15, 'min_lr': 1e-5, 'max_time': 12,
            'seed': 42,
        })
    else:  # eurosat
        params.update({
            'batch_size': 32, 'num_workers': 4, 'epochs': 500,
            'init_lr': 1e-3, 'weight_decay': 5e-4,
            'if_scheduler': True, 'gamma': 0.1,
            'ReduceLROnPlateau': True, 'lr_reduce_factor': 0.5,
            'lr_schedule_patience': 15, 'min_lr': 1e-10, 'max_time': 12,
            'seed': 42,
        })

    set_seed(params['seed'])
    return params


def train_pipeline(params):
    # Data setup
    transform = transforms.Compose([transforms.ToTensor()])
    if params['dataset'] == 'cifar10':
        train_ds = datasets.CIFAR10(root='../dataset/', train=True, download=True, transform=transform)
        test_ds = datasets.CIFAR10(root='../dataset/', train=False, download=True, transform=transform)
    elif params['dataset'] == 'imagenet':  # imagenet
        print("loading data of imagenet")
        train_dataset = datasets.ImageFolder(root='../dataset/ImageNet/train', transform=transform)

        train_loader = DataLoader(train_dataset, shuffle=True,
                                  batch_size=params['batch_size'], num_workers=params['num_workers'])
        test_dataset = Vanilla(root='../dataset/ImageNet/val', transform=transform)
        test_loader = DataLoader(test_dataset, shuffle=True,
                                 batch_size=params['batch_size'], num_workers=params['num_workers'])
    else: # eurosat
        print("loading data of eurosat")
        transform_resize = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()])
        # train_ds = datasets.ImageFolder('D:\Research\TiTiNguyen_SENTRY\Sentry_Data\\train', transform=transform_resize)
        # test_ds = datasets.ImageFolder('D:\Research\TiTiNguyen_SENTRY\Sentry_Data\\test', transform=transform_resize)
        train_ds = datasets.ImageFolder('/home/MATLAB_DATA/TiNguyen/Sentry_Data/train', transform=transform_resize)
        test_ds = datasets.ImageFolder('/home/MATLAB_DATA/TiNguyen/Sentry_Data/test', transform=transform_resize)

    train_loader = DataLoader(train_ds, batch_size=params['batch_size'],
                              shuffle=True, num_workers=params['num_workers'])
    test_loader = DataLoader(test_ds, batch_size=params['batch_size'],
                             shuffle=False, num_workers=params['num_workers'])

    # Model init
    sample_img, _ = train_ds[0]
    c = ratio2filtersize(sample_img, params['ratio'])
    if params['channel'] == 'Rician':
        print(f"🔧 {params['channel']}, K_factor={params['K_factor']}, SNR={params['snr']}, inner channel c={c}, ratio={params['ratio']:.2f}")
    else:
        print(f"🔧 {params['channel']}, SNR={params['snr']}, inner channel c={c}, ratio={params['ratio']:.2f}")

    model = DeepJSCC(c=c, channel_type=params['channel'], snr=params['snr'])
    model = setup_model_device(model, params)

    # Optimizer and scheduler
    optimizer = optim.Adam(model.parameters(), lr=params['init_lr'], weight_decay=params['weight_decay'])
    scheduler = setup_scheduler(optimizer, params)

    # Logging directories
    # phaser = f"{params['dataset'].upper()}_{c}_{params['snr']}_{params['ratio']:.2f}_{params['channel']}_{time.strftime('%Hh%Mm%Ss_on_%b_%d_%Y')}"
    if params['channel'] == 'Rician':
        phaser = f"{params['dataset'].upper()}_{c}_{params['snr']}_{params['ratio']:.2f}_{params['channel']}_{params['K_factor']}_Equz-{params['equalization']}"
    else:
        phaser = f"{params['dataset'].upper()}_{c}_{params['snr']}_{params['ratio']:.2f}_{params['channel']}_Equz-{params['equalization']}"
        

    log_dir = os.path.join(params['out_dir'], 'logs', phaser)
    ckpt_dir = os.path.join(params['out_dir'], 'checkpoints', phaser)
    os.makedirs(ckpt_dir, exist_ok=True)

    writer = SummaryWriter(log_dir=log_dir)
    writer.add_text('config', str(params))

    t_start = time.time()
    best_val = float('inf')

    try:
        for epoch in tqdm(range(params['epochs']), disable=params['disable_tqdm'], desc='Epoch'):
            t0 = time.time()
            train_loss = train_epoch(model, optimizer, params, train_loader)
            val_loss = evaluate_epoch(model, params, test_loader)

            writer.add_scalar('train_loss', train_loss, epoch)
            writer.add_scalar('val_loss', val_loss, epoch)
            writer.add_scalar('lr', optimizer.param_groups[0]['lr'], epoch)

            if epoch%100==0:
                tqdm.write(f"E{epoch} L_train={train_loss:.4f} L_val={val_loss:.4f} LR={optimizer.param_groups[0]['lr']:.4e} EpochTime={(time.time()-t0):.2f}s")

            # Save checkpoint
            torch.save(model.state_dict(), os.path.join(ckpt_dir, f"epoch_{epoch}.pth"))
            cleanup_checkpoints(ckpt_dir, keep_latest=2)

            # Scheduler step
            if params['ReduceLROnPlateau'] and scheduler:
                scheduler.step(val_loss)
            elif params['if_scheduler'] and scheduler:
                scheduler.step()

            # Early stop trigger
            if optimizer.param_groups[0]['lr'] < params['min_lr']:
                print("LR dropped below minimum threshold.")
                break

            # Optional: can implement early stopping here

            # Time check
            if time.time() - t_start > params['max_time'] * 3600:
                print("Max training time reached, exiting.")
                break

    except KeyboardInterrupt:
        print("Training interrupted by user.")

    # Final evaluation
    final_train = evaluate_epoch(model, params, train_loader)
    final_val = evaluate_epoch(model, params, test_loader)

    print(f"Done: Train Loss={final_train:.4f}, Val Loss={final_val:.4f}, TotalTime={(time.time()-t_start)/3600:.2f}h")

    # Save config YAML
    config_path = os.path.join(params['out_dir'], 'configs', phaser + '.yaml')
    os.makedirs(os.path.dirname(config_path), exist_ok=True)
    with open(config_path, 'w') as f:
        yaml.dump({'params': params, 'inner_channel': c, 'total_parameters': view_model_param(model)}, f)

    writer.close()


def setup_model_device(model, params):
    device = torch.device(params['device'])
    if params['parallel'] and torch.cuda.device_count() > 1:
        model = DataParallel(model).to(device)
    else:
        model = model.to(device)
    model.loss = model.module.loss if isinstance(model, DataParallel) else model.loss
    params['device'] = device
    params['parallel'] = params['parallel'] and torch.cuda.device_count() > 1
    return model


def setup_scheduler(optimizer, params):
    if not params['if_scheduler']:
        return None
    if params['ReduceLROnPlateau']:
        return optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min',
                                                    factor=params['lr_reduce_factor'],
                                                    patience=params['lr_schedule_patience'])
    else:
        return optim.lr_scheduler.StepLR(optimizer, step_size=params['step_size'], gamma=params['gamma'])


def cleanup_checkpoints(directory, keep_latest=2):
    files = sorted(glob.glob(os.path.join(directory, 'epoch_*.pth')), key=os.path.getmtime)
    old = files[:-keep_latest]
    for f in old:
        os.remove(f)


if __name__ == "__main__":
    main_pipeline()


Ignoring unknown args: ['--f=/run/user/1005/jupyter/runtime/kernel-v3516b31e20c13b2667e3a95f288634e867bda8ae5.json']
📡 Training Start
loading data of eurosat
🔧 Rician, K_factor=1.0, SNR=1.0, inner channel c=8, ratio=0.17


Epoch:   0%|          | 1/500 [00:04<36:13,  4.36s/it]

E0 L_train=980.7744 L_val=709.3661 LR=1.0000e-03 EpochTime=4.35s


Epoch:  20%|██        | 101/500 [06:30<25:39,  3.86s/it]

E100 L_train=121.6931 L_val=111.6050 LR=5.0000e-04 EpochTime=3.86s


...

Epoch: 100%|██████████| 500/500 [30:57<00:00,  3.72s/it]


Done: Train Loss=136.3233, Val Loss=132.7614, TotalTime=0.52h
loading data of eurosat
🔧 Rician, K_factor=3.0, SNR=4.0, inner channel c=4, ratio=0.08


Epoch:   0%|          | 1/500 [00:03<30:46,  3.70s/it]

E0 L_train=919.0705 L_val=582.1986 LR=1.0000e-03 EpochTime=3.69s


Epoch:  17%|█▋        | 87/500 [05:24<27:02,  3.93s/it]